# M04 — Induced prism forward physics and controls

**Level 3 · Prerequisites: foundations notebook; read time: 25 minutes.**

## Objectives

Run the product induced-prism forward operator on its frozen synthetic fixture. Compare components with the independent Choclo physical oracle and distinguish linear TMI from exact magnitude anomaly. This is not susceptibility inversion or remanence classification.

## Setup

Use the already reviewed Python 3.12 pipeline environment. This notebook installs nothing.

Import repository code and confirm its source file.

In [ ]:
import runpy,sys
from pathlib import Path
import numpy as np
root=Path.cwd().parents[2];sys.path.insert(0,str(root/'data-pipeline'))
import magnetic_forward
# Reuse the product's frozen synthetic physical control.
fixture=runpy.run_path(str(root/'tests/numerics/test_magnetic_forward.py'))
print(magnetic_forward.ENGINE)

## Data

Run the product induced-prism forward operator on its frozen synthetic fixture. Compare components with the independent Choclo physical oracle and distinguish linear TMI from exact magnitude anomaly. This is not susceptibility inversion or remanence classification.

**Files:** tests/numerics/test_magnetic_forward.py, docs/design/features/m04-induced-prism/algorithms.md, docs/design/features/m04-induced-prism/validation.md

## Method

For I positive down and D clockwise from north, f=(cos I sin D, cos I cos D, −sin I) in ENU; B₀=Ff. Uniform induced magnetization is M=χ·10⁻⁹B₀/μ₀ A m⁻¹. Secondary b gives linear TMI f·b; exact magnitude anomaly is |B₀+b|−F. Assumptions: uniform field, scalar susceptibility, no self-demagnetization and no remanence.

For inclination I and declination D, f=(cos I sin D, cos I cos D, −sin I) in east/north/up; Ff is the inducing field in nT. Scalar susceptibility χ is dimensionless and the ideal induced magnetization is M=χ·10⁻⁹Ff/μ₀ A m⁻¹. The secondary vector field integrates μ₀/(4π)[3(M·R)R/|R|⁵−M/|R|³] over each prism volume, then converts tesla to nT. Linear TMI is f·b. Exact magnitude anomaly is |Ff+b|−F, not |b|. Only linear TMI has the returned Jacobian with respect to χ. For |b|≪F, exact and linear anomalies approach one another; their difference is a diagnostic of approximation error.

Assumptions are uniform source field, scalar cellwise χ, no remanence, and no self-demagnetization. The processor does not infer those assumptions or classify a real survey. The added easting transect is a deterministic model experiment based on the product’s one-prism fixture. Its components are compared with an independent physical Choclo integral; the plotted exact-minus-linear curve shows the projection approximation for this case. A small forward-oracle difference validates only that controlled calculation, not inversion, field source validity or host acceptance.


Execute the product method on the declared control.

In [ ]:
request=fixture['request']()
# Compare actual product response and independent physical prism calculation.
result=magnetic_forward.forward_magnetic(request)
oracle=fixture['physical_oracle'](request)
max_error=float(np.max(np.abs(result['field_components_nt']-oracle)))
print({'components_nT':result['field_components_nt'].tolist(),'linear_TMI_nT':result['linear_tmi_nt'].tolist(),'exact_magnitude_nT':result['exact_magnitude_anomaly_nt'].tolist(),'max_component_error_nT':max_error,'accepted':result['linearization']['full_method_accepted']})

Adversarial or negative control: isolate one assumption or violate one declared physical constraint.

In [ ]:
from copy import deepcopy
changed=deepcopy(request);changed['inducing_field']['declination_deg']=42.0
# Change only a declared field assumption; this is not field classification.
alternative=magnetic_forward.forward_magnetic(changed)
delta=float(np.max(np.abs(alternative['field_components_nt']-result['field_components_nt'])))
print({'max_component_change_nT':delta})
assert delta>0

## State of the art

SimPEG/Geoana is the production framework ([Cockett et al., 2015](https://doi.org/10.1016/j.cageo.2015.09.015)); the product control compares it with an independent prism calculation. A changed field tests sensitivity but cannot identify the real field. Exact magnitude and linear TMI differ; the returned Jacobian applies to linear TMI only.

This deterministic synthetic transect uses the production operator and a separate physical oracle. The exact-minus-linear panel is an approximation diagnostic in nT, not a universal threshold or a measured anomaly.


In [ ]:
from copy import deepcopy
import matplotlib.pyplot as plt

# Recompute a physical easting transect with the product method and independent Choclo oracle.
transect=deepcopy(request); east=np.linspace(-500.,500.,81)
transect['receivers_m']=np.column_stack([east,np.zeros_like(east),np.full_like(east,100.)])
p=magnetic_forward.forward_magnetic(transect); oracle_components=fixture['physical_oracle'](transect)
err=float(np.max(np.abs(p['field_components_nt']-oracle_components)))
linear=p['linear_tmi_nt']; exact=p['exact_magnitude_anomaly_nt']
fig,(ax,bx)=plt.subplots(2,1,figsize=(8.5,7),sharex=True,constrained_layout=True)
ax.plot(east,linear,label='Linear TMI: f·b',lw=2); ax.plot(east,exact,'--',label='Exact: |B₀+b|−F',lw=1.6)
ax.set(ylabel='Total-field anomaly [nT]',title='Synthetic induced prism; receiver elevation z=100 m'); ax.legend(); ax.grid(alpha=.25)
bx.plot(east,exact-linear,color='#aa3377'); bx.axhline(0,color='.4',lw=.8)
bx.set(xlabel='Easting [m]',ylabel='Exact − linear [nT]'); bx.grid(alpha=.25)
print({'max_component_error_vs_oracle_nT':err,'max_exact_linear_difference_nT':p['linearization']['maximum_abs_difference_nt'],'max_secondary_to_background_ratio':float(np.max(p['linearization']['secondary_to_background_ratio'])),'full_method_accepted':p['linearization']['full_method_accepted']})
plt.show()


## Results

Use frozen independent tolerances, not rounded display values. A changed declination should change components. The operator sets field-source and full-method acceptance false; M04 survey inversion remains outside this lesson.

## Summary

Keep field convention, units, observables and independent oracle together. A forward image is not recovered geology.




## References

- https://doi.org/10.1016/j.cageo.2015.09.015